<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 00 Environment

The notebook checks the tools every other notebook relies on: the pinned toolchain with the target
of the Web Worker, wasm-bindgen of the version the lock of the worker pins, the submodule of Aosabi,
the devtool, the built module of WebAssembly and the built GraphQL engine, and the browser of the
interface notebooks. It fails first, so a missing tool or a stale build reads as such and not as a
broken feature. A cell that proves an entry of the register `Documentation/quirks.md` carries the
tag its column Test names, and its checks name the ID of the entry.

In [ ]:
require_build = True
require_browser = True  # tests/run.py passes False under --fast

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

## 1. The toolchain

In [ ]:
import re, subprocess
pin = re.search(r'channel = "([^"]+)"', (ROOT / 'rust-toolchain.toml').read_text()).group(1)
rustc = subprocess.run(['rustc', '--version'], cwd=ROOT, capture_output=True, text=True).stdout
check(pin in rustc, f'rustc is the pinned {pin}', rustc)
targets = subprocess.run(['rustup', 'target', 'list', '--installed'], cwd=ROOT, capture_output=True, text=True).stdout
check('wasm32-unknown-unknown' in targets, 'the target wasm32-unknown-unknown is installed', targets)

## 2. wasm-bindgen and the lock

In [ ]:
lock = (ROOT / 'crates' / 'bausteine-web' / 'Cargo.lock').read_text()
pinned = re.search(r'name = "wasm-bindgen"\nversion = "([^"]+)"', lock).group(1)
cli = subprocess.run(['wasm-bindgen', '--version'], capture_output=True, text=True).stdout
check(pinned in cli, f'wasm-bindgen-cli is the locked {pinned}', cli)

## 3. The submodule of Aosabi

The submodule `third_party/aosabi` carries the licence its own project chose in its file `LICENSE`,
and `Documentation/licensing.md` says how `cargo metadata` checks every crate a build links. The
file `.gitmodules` names the branch of Aosabi that adds the facade `aosabi::embed`, which the main
branch of Aosabi does not hold yet, so a fresh clone fetches that branch.

In [ ]:
facade = ROOT / 'third_party' / 'aosabi' / 'crates' / 'aosabi' / 'Cargo.toml'
check(facade.exists(), 'the submodule third_party/aosabi is checked out')
licence = ROOT / 'third_party' / 'aosabi' / 'LICENSE'
check(licence.exists() and licence.read_text(encoding='utf-8').strip() != '', 'R-48 the submodule carries the licence of its own project in its file LICENSE')
check('cargo metadata' in (ROOT / 'Documentation' / 'licensing.md').read_text(encoding='utf-8'),
      'R-48 Documentation/licensing.md says how cargo metadata checks the crates a build links')
modules = (ROOT / '.gitmodules').read_text(encoding='utf-8')
entry = re.search(r'\[submodule "third_party/aosabi"\]([^\[]*)', modules)
branch = re.search(r'^\s*branch = (\S+)$', entry.group(1), re.M) if entry else None
check(branch is not None, 'R-61 .gitmodules names the branch of Aosabi the submodule follows', modules)
print('the submodule follows the branch', branch.group(1))
lib = ROOT / 'third_party' / 'aosabi' / 'crates' / 'aosabi' / 'src' / 'lib.rs'
check((lib.parent / 'embed.rs').exists() and re.search(r'^pub mod embed;', lib.read_text(encoding='utf-8'), re.M) is not None,
      'R-61 the submodule holds the facade aosabi::embed, which the main branch of Aosabi lacks')

## 4. The devtool and the builds

In [ ]:
build_devtool()
check(len(devtool('examples')) == 5, 'the devtool answers the five examples')
if require_build:
    check((ROOT / 'web' / 'pkg' / 'bausteine_web_bg.wasm').exists(), 'the module of WebAssembly is built into web/pkg')
    check(ENGINE.exists(), 'the GraphQL engine bausteine-graphql is built')
    check((ROOT / 'web' / 'js' / 'worker.js').exists(), 'the script of the Web Worker is there')

## 5. The browser of the interface notebooks

The Web Worker boots the service through the facade `aosabi::embed`, which the submodule holds on
its own branch. The notebooks 60 to 90, and the sections of the engine notebooks that open the page,
drive the page in headless Chromium through the Playwright that `tests/requirements.txt` pins. The
page loads the module of WebAssembly that `tools/build.sh` writes into `web/pkg`, and the notebook
50 asks the engine `bausteine-graphql`, so each must be newer than every source it is built from:
the files cargo records in the file of dependencies beside the product, and the sources of the
service, of which `cargo aosabi build` writes the copy that cargo compiles. Under `--fast` the run
has no browser, and the checks of the browser and of the module stand aside, since `tests/run.py`
leaves the notebooks 60 to 90 out. Headless Chromium on SwiftShader loses its device of WebGPU at
the first present to a canvas, so the harness opens the page with `?offscreen`, and the worker then
draws into a texture of its own, which `snapshot` reads back.

In [ ]:
import sys
from importlib import metadata
embed = ROOT / 'third_party' / 'aosabi' / 'crates' / 'aosabi' / 'src' / 'embed.rs'
check(embed.exists(), 'the submodule holds the facade crates/aosabi/src/embed.rs the worker boots the service with')

import importlib.util, inspect
spec = importlib.util.spec_from_file_location('runner', ROOT / 'tests' / 'run.py')
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
books = sorted(runner.NOTEBOOKS.glob('*.ipynb'))
interface = [b.name for b in books if b.name.startswith(runner.BROWSER)]
check(runner.BROWSER == ('60_', '70_', '80_', '90_') and interface == [b.name for b in books if b.name[:2] in ('60', '70', '80', '90')],
      'R-62 tests/run.py counts the notebooks 60 to 90 as those of the browser, which --fast leaves out', interface)
check(all(runner.parameters(b, True).get('require_browser', False) is False for b in books if b.name.startswith(('00_', '20_'))),
      'R-62 under --fast the run leaves the browser out and hands require_browser False to the engine notebooks')
opened = inspect.getsource(Browser.open)
worker_js = (ROOT / 'web' / 'js' / 'worker.js').read_text(encoding='utf-8')
main_js = (ROOT / 'web' / 'js' / 'main.js').read_text(encoding='utf-8')
check('?debug&offscreen' in opened and "offscreen: params.has('offscreen')" in main_js and 'if (m.offscreen) drawOffscreen();' in worker_js,
      'R-53 the notebooks open the page with ?offscreen, and the worker then draws into a texture of its own that snapshot reads back')

WEB_TARGET = ROOT / 'crates' / 'bausteine-web' / 'target'
SERVICE = [p for p in (ROOT / 'service').rglob('*') if p.is_file() and 'target' not in p.relative_to(ROOT / 'service').parts]

def recorded(depfile):
    # The sources cargo records for a product, the first rule of its file of dependencies, blanks escaped as make does.
    first = depfile.read_text().splitlines()[0]
    deps = first.split(': ', 1)[1] if ': ' in first else ''
    return [Path(d.replace('\\ ', ' ')) for d in re.split(r'(?<!\\) ', deps) if d]

def newest_source(depfile):
    sources = [p for p in recorded(depfile) + SERVICE if p.exists()]
    return max(sources, key=lambda p: p.stat().st_mtime)

def check_fresh(product, depfile, what):
    check(product.exists() and depfile.exists(), f'{what} is built with its file of dependencies', product, depfile)
    newest = newest_source(depfile)
    check(product.stat().st_mtime >= newest.stat().st_mtime,
          f'{what} is newer than every source it is built from, else run sh tools/build.sh --dev', f'newer: {newest}')

if require_build:
    check_fresh(ENGINE, WEB_TARGET / 'debug' / 'bausteine-graphql.d', 'the engine bausteine-graphql')
if require_build and require_browser:
    # The profile tools/build.sh built the module with last is the one web/pkg holds.
    built = max((WEB_TARGET / 'wasm32-unknown-unknown' / p / 'bausteine_web.wasm' for p in ('debug', 'release')),
                key=lambda p: p.stat().st_mtime if p.exists() else 0)
    wasm = ROOT / 'web' / 'pkg' / 'bausteine_web_bg.wasm'
    check_fresh(wasm, built.with_suffix('.d'), 'the module of WebAssembly in web/pkg')
    check(wasm.stat().st_mtime >= built.stat().st_mtime, 'R-62 web/pkg holds the module cargo built last, bound by wasm-bindgen')
    pins = dict(line.split('==') for line in (ROOT / 'tests' / 'requirements.txt').read_text().splitlines() if '==' in line)
    check(metadata.version('playwright') == pins['playwright'], f"Playwright is the pinned {pins['playwright']}", metadata.version('playwright'))
    named = chromium()
    if named is None:
        probe = subprocess.run([sys.executable, '-c', 'from playwright.sync_api import sync_playwright\n'
                                'p = sync_playwright().start()\nprint(p.chromium.executable_path)\np.stop()'],
                               capture_output=True, text=True)
        named = probe.stdout.strip()
    check(bool(named) and Path(named).exists(), f'R-62 the Chromium of the interface notebooks is at {named}')
else:
    print('the run has no browser, the checks of the browser and of the module stand aside')